# Пакетная обработка CSV/TSV через API: чанкинг, ретраи и контроль квоты

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DimaKorshikov/rulemma/blob/main/examples/notebooks/05_batch_csv_pipeline_and_rate_limits.ipynb)

Готовый шаблон для боевой обработки файлов с тысячами запросов через HTTP API `rulemma.ru`: автоматическое чтение `max_batch` и `rate_limit_rps` из `GET /v1/usage`, разбиение на чанки, вежливая обработка `429 Retry-After` и `5xx`, контроль остатка квоты и запись обогащённого TSV/CSV.

### Регистрация и доступ к HTTP API

- Личный кабинет и бесплатный API-ключ: **[rulemma.ru/account/](https://rulemma.ru/account/)** (вход по email, Яндекс ID или Google)
- Документация HTTP API: [rulemma.ru/api/](https://rulemma.ru/api/) · Тарифы и лимиты: [rulemma.ru/pricing/](https://rulemma.ru/pricing/) · OpenAPI: [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Базовый адрес API: `https://rulemma.ru/v1` · Авторизация: заголовок `Authorization: Bearer rlm_live_...` (передавать ключ в URL запрещено)

Как подключить ключ:
1. Создайте бесплатный аккаунт на [rulemma.ru/account/](https://rulemma.ru/account/) и скопируйте токен `rlm_live_...`.
2. Сохраните его в переменную окружения `RULEMMA_API_KEY` (в Google Colab удобно положить токен в панель `Secrets` под именем `RULEMMA_API_KEY` или ввести в ячейке настройки).
3. Для реальных файлов с тысячами строк нужен API-ключ ([rulemma.ru/account/](https://rulemma.ru/account/)). Без ключа демонстрационный файл из 8 строк пройдёт через песочницу, чтобы можно было проверить весь пайплайн целиком.

### Содержание тетрадки

1. Подключение и чтение лимитов аккаунта из `GET /v1/usage`
2. Функция батчевой отправки с ретраями (`Retry-After` / backoff) и троттлингом
3. Подготовка демонстрационного TSV-файла
4. Потоковая обработка TSV чанками и сохранение результата
5. Сводка расхода единиц и контроль остатка квоты

## 1. Клиент HTTP API

In [ ]:
import json
import os
import urllib.error
import urllib.request

BASE_URL = os.environ.get("RULEMMA_API_BASE", "https://rulemma.ru/v1").rstrip("/")
API_KEY = os.environ.get("RULEMMA_API_KEY", "").strip()

# В Google Colab ключ можно автоматически прочитать из вкладки Secrets
if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore

        API_KEY = (userdata.get("RULEMMA_API_KEY") or "").strip()
    except Exception:
        pass

HAS_KEY = bool(API_KEY)
print("Базовый URL:", BASE_URL)
print(
    "Режим:",
    "боевой API с Bearer-токеном" if HAS_KEY else "публичная песочница без ключа (до 10 строк)",
)
if not HAS_KEY:
    print("Получить бесплатный API-ключ: https://rulemma.ru/account/")


def api_request(
    method: str,
    path: str,
    payload: dict | None = None,
    *,
    auth: bool = True,
    timeout: int = 20,
) -> tuple[dict, dict[str, str]]:
    """Выполняет HTTP-запрос к rulemma API без сторонних зависимостей."""
    url = f"{BASE_URL}{path}"
    headers = {
        "Accept": "application/json",
        "User-Agent": "rulemma-jupyter-example/1.0",
    }
    if auth:
        if not API_KEY:
            raise RuntimeError(
                "Требуется RULEMMA_API_KEY. Выпустите бесплатный токен на https://rulemma.ru/account/"
            )
        headers["Authorization"] = f"Bearer {API_KEY}"

    body = None
    if payload is not None:
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json; charset=utf-8"

    import time as _time

    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    for attempt in range(2):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                resp_headers = {k.lower(): v for k, v in resp.headers.items()}
                return json.loads(resp.read().decode("utf-8")), resp_headers
        except urllib.error.HTTPError as exc:
            raw = exc.read().decode("utf-8", errors="replace")
            if exc.code == 429 and attempt == 0:
                wait = min(12, max(1, int(exc.headers.get("Retry-After", "4"))))
                print(f"Лимит скорости (HTTP 429), пауза {wait} с...")
                _time.sleep(wait)
                continue
            raise RuntimeError(f"HTTP {exc.code} {path}: {raw}") from exc
    raise RuntimeError(f"Не удалось выполнить {method} {path}")

## 2. Определение безопасного размера чанка и лимита скорости

Не хардкодьте лимиты: один вызов `GET /v1/usage` сразу возвращает актуальные `max_batch`, `rate_limit_rps` и остаток квоты вашего тарифа.

In [ ]:
import time

if HAS_KEY:
    usage_before, _ = api_request("GET", "/usage")
    CHUNK_SIZE = min(2000, max(1, int(usage_before["max_batch"])))
    MIN_INTERVAL_SEC = 1.0 / max(1, int(usage_before["rate_limit_rps"]))
    print(
        f"Тариф: {usage_before['plan_name']} | чанк: {CHUNK_SIZE} | "
        f"мин. интервал: {MIN_INTERVAL_SEC:.3f} с | "
        f"остаток: {usage_before['quota']['remaining']:,}"
    )
else:
    CHUNK_SIZE = 10
    MIN_INTERVAL_SEC = 0.5
    print("Режим без ключа (песочница): чанк до 10 строк.")
    print("Для батчей до 3 000+ строк получите бесплатный ключ: https://rulemma.ru/account/")

## 3. Надёжная отправка чанка с ретраями и чтением `Retry-After`

При `429` (rate limit) или временных `502`/`503`/`504` клиент ждёт паузу из заголовка `Retry-After` (или экспоненциальный backoff) и повторяет попытку без потери строк.

In [ ]:
def lemmatize_chunk_with_retries(
    queries: list[str],
    *,
    profile: str = "technical",
    max_retries: int = 4,
) -> tuple[list[dict], int, str]:
    """Обрабатывает один чанк строк; возвращает (items, units_charged, quota_remaining)."""
    payload = {
        "queries": queries,
        "profile": profile,
        "normalize_homoglyphs": True,
    }
    for attempt in range(1, max_retries + 1):
        try:
            if HAS_KEY:
                data, hdrs = api_request("POST", "/lemmatize", payload)
                return (
                    data["items"],
                    int(data.get("units_charged", len(queries))),
                    hdrs.get("x-quota-remaining", "?"),
                )
            data, _ = api_request(
                "POST",
                "/sandbox/test",
                {"method": "lemmatize", **payload},
                auth=False,
            )
            rem = data.get("limits", {}).get("daily_limit", 100) - data.get("limits", {}).get(
                "daily_used", 0
            )
            return data["items"], 0, f"sandbox({rem} req/day)"
        except RuntimeError as exc:
            msg = str(exc)
            is_retryable = any(code in msg for code in ("HTTP 429", "HTTP 502", "HTTP 503", "HTTP 504"))
            if not is_retryable or attempt == max_retries:
                raise
            wait_sec = min(16.0, 1.5 * (2 ** (attempt - 1)))
            print(f"Временный ответ сервера (попытка {attempt}/{max_retries}), пауза {wait_sec:.1f} с...")
            time.sleep(wait_sec)
    raise RuntimeError("Не удалось обработать чанк")

## 4. Обработка TSV-файла чанками и сохранение результата

Создадим небольшой тестовый TSV (в реальной работе просто укажите путь к своему файлу) и обогатим его тремя колонками: `lemmatized`, `lemma_key`, `semantic_key`.

In [ ]:
import csv
import tempfile
from pathlib import Path

demo_rows = [
    ("1", "зимние кроссовки найк", "12400"),
    ("2", "кроссовки на зиму найк", "8900"),
    ("3", "бензопила 2,6 л.с. б/у", "3100"),
    ("4", "бензопила 6,2 л.с.", "950"),
    ("5", "плитка 100×100 см белая", "4200"),
    ("6", "кабель 5cм intеl рrо", "610"),
    ("7", "шоколад без сахара", "18500"),
    ("8", "шоколад с сахаром", "14200"),
]

work_dir = Path(tempfile.mkdtemp(prefix="rulemma_api_nb_"))
in_tsv = work_dir / "queries_in.tsv"
out_tsv = work_dir / "queries_out.tsv"

with in_tsv.open("w", encoding="utf-8", newline="") as f:
    w = csv.writer(f, delimiter="\t")
    w.writerow(["id", "query", "impressions"])
    w.writerows(demo_rows)

total_rows = 0
total_units = 0
last_remaining = "?"
t0 = time.perf_counter()

with (
    in_tsv.open("r", encoding="utf-8", newline="") as fin,
    out_tsv.open("w", encoding="utf-8", newline="") as fout,
):
    reader = csv.DictReader(fin, delimiter="\t")
    fieldnames = list(reader.fieldnames or []) + ["lemmatized", "lemma_key", "semantic_key"]
    writer = csv.DictWriter(fout, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()

    batch_rows: list[dict] = []

    def flush_batch() -> None:
        global total_rows, total_units, last_remaining
        if not batch_rows:
            return
        texts = [r["query"] for r in batch_rows]
        items, charged, rem = lemmatize_chunk_with_retries(texts)
        for src_row, res in zip(batch_rows, items):
            src_row["lemmatized"] = res["lemmatized"]
            src_row["lemma_key"] = res["lemma_key"]
            src_row["semantic_key"] = res["semantic_key"]
            writer.writerow(src_row)
        total_rows += len(batch_rows)
        total_units += charged
        last_remaining = rem
        batch_rows.clear()
        time.sleep(MIN_INTERVAL_SEC)

    for row in reader:
        batch_rows.append(row)
        if len(batch_rows) >= CHUNK_SIZE:
            flush_batch()
    flush_batch()

elapsed = time.perf_counter() - t0
print(f"Готово: {total_rows} строк за {elapsed:.2f} с | списано ед.: {total_units} | остаток: {last_remaining}")
print(f"Файл результата: {out_tsv}\n")
print(out_tsv.read_text(encoding="utf-8"))

---

## Что дальше

- Все Jupyter-тетрадки с HTTP API: [examples/notebooks/README.md](README.md)
- Регистрация и управление API-ключами: [rulemma.ru/account/](https://rulemma.ru/account/)
- Документация эндпоинтов и форматов ошибок: [rulemma.ru/api/](https://rulemma.ru/api/) · [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Нужны миллионы строк локально без сети (Go / Python CGO / CLI)? Смотрите локальные примеры [examples/01..13](../README.md).